# 11 — ODIR Protocol Sanity Audit

## NO TRAINING

This notebook determines whether the previous poor ODIR zero-shot result is caused by:

- a real cross-domain generalisation failure,
- an annotation/subset construction problem,
- an image-representation/preprocessing problem,
- or a mismatch with the historical saved ODIR protocol.

It performs:

1. Historical saved ODIR artifact recovery.
2. `full_df.csv` image-level audit using its own `filename` field.
3. Conservative Cataract-v-Normal subset construction.
4. Separate RAW and `preprocessed_images` matching.
5. Zero-shot evaluation on both representations.
6. Historical-vs-corrected comparison.
7. A final protocol verdict.

No model is trained or fine-tuned.

In [ ]:
# ============================================================
# CELL 1 — SETUP
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import re
import sys
import subprocess

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import confusion_matrix, roc_auc_score

PROJECT = Path('/content/drive/MyDrive/Cataract')

FINAL_ROOT = PROJECT / 'FINAL_REVISION_2026_08'

MODEL_PATH = (
    FINAL_ROOT
    / 'reviewer_10_2_clean_split'
    / 'MobileNetV2_frozen'
    / 'best.keras'
)

OLD_EXT = PROJECT / 'external_validation'

OUT = FINAL_ROOT / 'odir_protocol_sanity_audit'
OUT.mkdir(parents=True, exist_ok=True)

SEED = 42
BOOTSTRAPS = 5000
IMAGE_SIZE = (224, 224)

assert PROJECT.exists(), f'STOP: Missing project: {PROJECT}'
assert MODEL_PATH.exists(), f'STOP: Missing final model: {MODEL_PATH}'

print('TensorFlow:', tf.__version__)
print('Project:', PROJECT)
print('Model:', MODEL_PATH)
print('Historical external folder:', OLD_EXT)
print('Output:', OUT)
print('\n✅ CELL 1 COMPLETE')

In [ ]:
# ============================================================
# CELL 2 — RECOVER HISTORICAL SAVED ODIR RESULT
# ============================================================

old_paths = {
    'metrics': OLD_EXT / 'ODIR5K_metrics.csv',
    'y_true': OLD_EXT / 'ODIR5K_y_true.npy',
    'y_pred': OLD_EXT / 'ODIR5K_y_pred.npy',
    'y_probs': OLD_EXT / 'ODIR5K_y_probs.npy',
    'multisource': OLD_EXT / 'MultiSource_Results.csv'
}

print('Historical artifact existence:')
for k, p in old_paths.items():
    print(f'{k:12s}: {p.exists()}  {p}')

missing_core = [
    k for k in ['y_true', 'y_pred', 'y_probs']
    if not old_paths[k].exists()
]

if missing_core:
    raise RuntimeError(
        'STOP: Missing historical ODIR prediction artifacts: '
        + ', '.join(missing_core)
    )

old_y_true = np.load(old_paths['y_true'])
old_y_pred = np.load(old_paths['y_pred'])
old_probs = np.load(old_paths['y_probs'])

print('\nHistorical shapes:')
print('y_true:', old_y_true.shape, old_y_true.dtype)
print('y_pred:', old_y_pred.shape, old_y_pred.dtype)
print('probs :', old_probs.shape, old_probs.dtype)

u, c = np.unique(old_y_true, return_counts=True)
print('\nHistorical y_true counts:', dict(zip(u.tolist(), c.tolist())))

u, c = np.unique(old_y_pred, return_counts=True)
print('Historical y_pred counts:', dict(zip(u.tolist(), c.tolist())))

if old_probs.ndim == 2:
    print('Mean probs:', np.mean(old_probs, axis=0))

if old_paths['metrics'].exists():
    print('\nSaved ODIR metrics:')
    display(pd.read_csv(old_paths['metrics']))

if old_paths['multisource'].exists():
    print('\nSaved MultiSource results:')
    display(pd.read_csv(old_paths['multisource']))

assert len(old_y_true) == len(old_y_pred) == len(old_probs)

if old_probs.ndim == 2:
    print(
        'y_pred equals argmax(probs):',
        np.array_equal(old_y_pred, np.argmax(old_probs, axis=1))
    )

print('\n✅ CELL 2 COMPLETE')

In [ ]:
# ============================================================
# CELL 3 — RECOMPUTE HISTORICAL METRICS
# ============================================================

def binary_metrics(y_binary, pred_binary, score):
    y_binary = np.asarray(y_binary, dtype=int)
    pred_binary = np.asarray(pred_binary, dtype=int)
    score = np.asarray(score, dtype=float)

    cm = confusion_matrix(y_binary, pred_binary, labels=[0, 1])
    TN, FP, FN, TP = cm.ravel()

    accuracy = (TP + TN) / len(y_binary)
    sensitivity = TP / (TP + FN) if (TP + FN) else np.nan
    specificity = TN / (TN + FP) if (TN + FP) else np.nan

    auc = (
        roc_auc_score(y_binary, score)
        if len(np.unique(y_binary)) == 2
        else np.nan
    )

    return {
        'N': int(len(y_binary)),
        'Accuracy': float(accuracy),
        'Sensitivity': float(sensitivity),
        'Specificity': float(specificity),
        'AUC': float(auc),
        'TN': int(TN),
        'FP': int(FP),
        'FN': int(FN),
        'TP': int(TP)
    }

rows = [{
    'Interpretation': 'Saved_y_pred_equals_y_true',
    'N': len(old_y_true),
    'Accuracy': float(np.mean(old_y_true == old_y_pred))
}]

old_binary_result = None

if (
    set(np.unique(old_y_true)).issubset({0, 1})
    and old_probs.ndim == 2
    and old_probs.shape[1] >= 2
):
    old_y_binary = (old_y_true == 0).astype(int)

    denom = old_probs[:, 0] + old_probs[:, 1]

    old_score = np.divide(
        old_probs[:, 0],
        denom,
        out=np.full(len(old_probs), 0.5, dtype=float),
        where=denom > 0
    )

    old_pred_binary = (old_score >= 0.5).astype(int)

    old_binary_result = binary_metrics(
        old_y_binary,
        old_pred_binary,
        old_score
    )

    old_binary_result['Interpretation'] = (
        'Assume saved class 0=Cataract, class 1=Normal'
    )

    rows.append(old_binary_result)

old_recomputed_df = pd.DataFrame(rows)

old_recomputed_df.to_csv(
    OUT / 'Historical_ODIR_Recomputed_Metrics.csv',
    index=False
)

display(old_recomputed_df)

print('\n✅ CELL 3 COMPLETE')

In [ ]:
# ============================================================
# CELL 4 — RECOVER + INSPECT full_df.csv
# ============================================================

candidate_full_df = list(PROJECT.rglob('full_df.csv'))

if not candidate_full_df:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q', '-U', 'kagglehub'
    ])

    import kagglehub

    returned = Path(
        kagglehub.dataset_download(
            'andrewmvd/ocular-disease-recognition-odir5k',
            path='full_df.csv'
        )
    )

    if returned.is_file():
        candidate_full_df.append(returned)
    elif returned.is_dir():
        candidate_full_df += list(returned.rglob('full_df.csv'))

if not candidate_full_df:
    raise RuntimeError('STOP: full_df.csv could not be recovered.')

FULL_DF_PATH = candidate_full_df[0]
full_df = pd.read_csv(FULL_DF_PATH)

print('full_df source:', FULL_DF_PATH)
print('Rows:', len(full_df))
print('Columns:', len(full_df.columns))
print('\nColumns:')
for c in full_df.columns:
    print(' -', c)

required = [
    'ID',
    'Left-Fundus',
    'Right-Fundus',
    'Left-Diagnostic Keywords',
    'Right-Diagnostic Keywords',
    'filename'
]

missing = [c for c in required if c not in full_df.columns]

if missing:
    raise RuntimeError(
        'STOP: Required full_df columns missing: '
        + ', '.join(missing)
    )

print('\nfilename unique:', full_df['filename'].nunique())
print('filename duplicate rows:', int(full_df['filename'].duplicated().sum()))

inspect_cols = [
    c for c in [
        'ID',
        'filename',
        'labels',
        'target',
        'Left-Fundus',
        'Right-Fundus',
        'Left-Diagnostic Keywords',
        'Right-Diagnostic Keywords'
    ]
    if c in full_df.columns
]

display(full_df[inspect_cols].head(20))

for col in ['labels', 'target']:
    if col in full_df.columns:
        print(f'\n{col} common values:')
        display(full_df[col].value_counts(dropna=False).head(20))

print('\n✅ CELL 4 COMPLETE')

In [ ]:
# ============================================================
# CELL 5 — BUILD CORRECT IMAGE-LEVEL CATARACT-v-NORMAL SUBSET
# ============================================================

def norm_name(x):
    return Path(str(x).strip()).name.lower()

def strict_eye_label(diagnostic):
    s = str(diagnostic).strip().lower()

    if 'cataract' in s:
        return 'Cataract'

    if s in {'normal fundus', 'normal', 'normal fundus photo'}:
        return 'Normal'

    return 'Exclude'

rows = []
side_mismatch_rows = []

for idx, r in full_df.iterrows():
    filename = str(r['filename']).strip()
    fkey = norm_name(filename)

    left = norm_name(r['Left-Fundus'])
    right = norm_name(r['Right-Fundus'])

    if fkey == left:
        side = 'Left'
        diagnostic = str(r['Left-Diagnostic Keywords'])
    elif fkey == right:
        side = 'Right'
        diagnostic = str(r['Right-Diagnostic Keywords'])
    else:
        side_mismatch_rows.append({
            'row_index': idx,
            'ID': r['ID'],
            'filename': filename,
            'Left-Fundus': r['Left-Fundus'],
            'Right-Fundus': r['Right-Fundus']
        })
        continue

    row_out = {
        'row_index': idx,
        'patient_id': str(r['ID']).strip(),
        'eye_side': side,
        'filename': Path(filename).name,
        'filename_key': fkey,
        'diagnostic_keyword': diagnostic,
        'external_label': strict_eye_label(diagnostic)
    }

    if 'labels' in full_df.columns:
        row_out['full_df_labels'] = r['labels']

    if 'target' in full_df.columns:
        row_out['full_df_target'] = r['target']

    rows.append(row_out)

image_level = pd.DataFrame(rows)
side_mismatch_df = pd.DataFrame(side_mismatch_rows)

image_level.to_csv(
    OUT / 'full_df_Image_Level_Audit.csv',
    index=False
)

side_mismatch_df.to_csv(
    OUT / 'full_df_Filename_Side_Mismatches.csv',
    index=False
)

print('full_df rows:', len(full_df))
print('Rows mapped to eye side:', len(image_level))
print('Side mismatches:', len(side_mismatch_df))

print('\nImage-level label counts:')
display(
    image_level['external_label']
    .value_counts()
    .rename_axis('Label')
    .reset_index(name='Images')
)

strict_subset = image_level[
    image_level['external_label'].isin(['Cataract', 'Normal'])
].copy()

print('\nStrict subset rows:', len(strict_subset))
print('Strict unique filenames:', strict_subset['filename_key'].nunique())
print('Strict patients:', strict_subset['patient_id'].nunique())

label_nunique = (
    strict_subset
    .groupby('filename_key')['external_label']
    .nunique()
)

conflict_keys = set(label_nunique[label_nunique > 1].index)

print('Filename label conflicts:', len(conflict_keys))

if conflict_keys:
    raise RuntimeError(
        'STOP: Same filename has conflicting strict labels.'
    )

strict_unique = (
    strict_subset
    .sort_values(['filename_key', 'patient_id'])
    .drop_duplicates(subset=['filename_key'], keep='first')
    .reset_index(drop=True)
)

strict_unique.to_csv(
    OUT / 'ODIR_Strict_ImageLevel_Cataract_Normal.csv',
    index=False
)

print('\nFINAL STRICT IMAGE-LEVEL SUBSET:')
print('N:', len(strict_unique))
print('Patients:', strict_unique['patient_id'].nunique())

display(
    strict_unique['external_label']
    .value_counts()
    .rename_axis('Class')
    .reset_index(name='Images')
)

print('\nHistorical saved ODIR N:', len(old_y_true))
print(
    'Difference strict N - historical N:',
    len(strict_unique) - len(old_y_true)
)

print('\n✅ CELL 5 COMPLETE')

In [ ]:
# ============================================================
# CELL 6 — LOCATE RAW AND PREPROCESSED ODIR IMAGES SEPARATELY
# ============================================================

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '-U', 'kagglehub'
])

import kagglehub

KAGGLE_ROOT = Path(
    kagglehub.dataset_download(
        'andrewmvd/ocular-disease-recognition-odir5k'
    )
)

assert KAGGLE_ROOT.exists(), (
    f'STOP: Kaggle root does not exist: {KAGGLE_ROOT}'
)

print('Kaggle ODIR root:', KAGGLE_ROOT)

IMAGE_SUFFIXES = {
    '.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'
}

all_images = [
    p for p in KAGGLE_ROOT.rglob('*')
    if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES
]

raw_images = []
pre_images = []
other_images = []

for p in all_images:
    low = str(p).lower()

    if 'preprocessed_images' in low:
        pre_images.append(p)
    elif (
        'training images' in low
        or 'training_images' in low
        or 'testing images' in low
        or 'testing_images' in low
        or '/training/' in low
        or '/testing/' in low
    ):
        raw_images.append(p)
    else:
        other_images.append(p)

print('Total images:', len(all_images))
print('Raw/training/testing:', len(raw_images))
print('Preprocessed:', len(pre_images))
print('Other:', len(other_images))

def basename_map(paths):
    d = {}
    for p in paths:
        d.setdefault(p.name.lower(), []).append(p)
    return d

raw_map = basename_map(raw_images)
pre_map = basename_map(pre_images)
other_map = basename_map(other_images)

audit_rows = []

for _, r in strict_unique.iterrows():
    key = r['filename_key']

    raw_matches = raw_map.get(key, [])
    pre_matches = pre_map.get(key, [])
    other_matches = other_map.get(key, [])

    audit_rows.append({
        **r.to_dict(),
        'raw_count': len(raw_matches),
        'preprocessed_count': len(pre_matches),
        'other_count': len(other_matches),
        'raw_filepath': str(raw_matches[0]) if raw_matches else '',
        'preprocessed_filepath': str(pre_matches[0]) if pre_matches else '',
        'other_filepath': str(other_matches[0]) if other_matches else ''
    })

representation_audit = pd.DataFrame(audit_rows)

representation_audit.to_csv(
    OUT / 'ODIR_Raw_vs_Preprocessed_File_Audit.csv',
    index=False
)

print('\nRepresentation coverage:')

for col in ['raw_filepath', 'preprocessed_filepath', 'other_filepath']:
    n = int(representation_audit[col].astype(str).ne('').sum())
    print(
        f'{col:24s}: '
        f'{n}/{len(representation_audit)} '
        f'({100*n/len(representation_audit):.2f}%)'
    )

common = representation_audit[
    representation_audit['raw_filepath'].astype(str).ne('')
    &
    representation_audit['preprocessed_filepath'].astype(str).ne('')
].copy()

common.to_csv(
    OUT / 'ODIR_Common_Raw_Preprocessed_Subset.csv',
    index=False
)

print('\nCommon raw+preprocessed filenames:', len(common))
print('\n✅ CELL 6 COMPLETE')

In [ ]:
# ============================================================
# CELL 7 — SEARCH OLD CODE FOR ODIR PROTOCOL CLUES
# ============================================================

TEXT_SUFFIXES = {'.py', '.ipynb', '.txt', '.md', '.json'}

SEARCH_TERMS = [
    'ODIR5K_y_true',
    'ODIR5K_y_probs',
    'ODIR5K_metrics',
    'MultiSource_Results',
    'full_df.csv'
]

hits = []

for p in PROJECT.rglob('*'):
    if not p.is_file():
        continue

    if p.suffix.lower() not in TEXT_SUFFIXES:
        continue

    try:
        p.resolve().relative_to(OUT.resolve())
        continue
    except Exception:
        pass

    try:
        text = p.read_text(errors='ignore')
    except Exception:
        continue

    for term in SEARCH_TERMS:
        idx = text.find(term)

        if idx >= 0:
            start = max(0, idx - 500)
            end = min(len(text), idx + 1500)

            hits.append({
                'file': str(p),
                'term': term,
                'snippet': text[start:end]
            })

protocol_hits = pd.DataFrame(hits)

protocol_hits.to_csv(
    OUT / 'Historical_ODIR_Code_Search_Hits.csv',
    index=False
)

print('Protocol/code hits:', len(protocol_hits))

if len(protocol_hits):
    display(protocol_hits.head(50))
else:
    print('No source-code hits found.')

print('\n✅ CELL 7 COMPLETE')

In [ ]:
# ============================================================
# CELL 8 — LOAD FINAL MODEL + EVALUATION HELPER
# ============================================================

model = tf.keras.models.load_model(
    MODEL_PATH,
    compile=False
)

print('Loaded final MobileNetV2.')
print('Layers:', len(model.layers))

def predict_filepaths(filepaths, batch_size=32):
    filepaths = list(map(str, filepaths))
    probs_out = []

    for start in range(0, len(filepaths), batch_size):
        batch_paths = filepaths[start:start + batch_size]
        batch = []

        for fp in batch_paths:
            img = tf.keras.utils.load_img(
                fp,
                target_size=IMAGE_SIZE,
                color_mode='rgb',
                interpolation='nearest'
            )

            arr = tf.keras.utils.img_to_array(img).astype(np.float32)
            arr = arr / 255.0
            batch.append(arr)

        batch = np.stack(batch, axis=0)

        p = model(
            batch,
            training=False
        ).numpy()

        probs_out.append(p)

    return np.concatenate(probs_out, axis=0)

def evaluate_external(df, filepath_col, label):
    if len(df) == 0:
        return pd.DataFrame(), {}

    probs = predict_filepaths(df[filepath_col].tolist())

    y = (
        df['external_label'].to_numpy()
        == 'Cataract'
    ).astype(int)

    denom = probs[:, 0] + probs[:, 1]

    score = np.divide(
        probs[:, 0],
        denom,
        out=np.full(len(probs), 0.5, dtype=float),
        where=denom > 0
    )

    pred = (score >= 0.5).astype(int)

    result = binary_metrics(y, pred, score)
    result['Representation'] = label

    result['ThreeClass_NotEye_Rate'] = float(
        (np.argmax(probs, axis=1) == 2).mean()
    )

    result['Mean_P_Cataract'] = float(np.mean(probs[:, 0]))
    result['Mean_P_Normal'] = float(np.mean(probs[:, 1]))
    result['Mean_P_NotEye'] = float(np.mean(probs[:, 2]))

    out_df = df.copy()

    out_df['P_Cataract'] = probs[:, 0]
    out_df['P_Normal'] = probs[:, 1]
    out_df['P_NotEye'] = probs[:, 2]
    out_df['Clinical_Cataract_Score'] = score
    out_df['y_true_binary'] = y
    out_df['y_pred_binary'] = pred

    return out_df, result

print('\n✅ CELL 8 COMPLETE')

In [ ]:
# ============================================================
# CELL 9 — ZERO-SHOT RAW ODIR
# ============================================================

raw_df = representation_audit[
    representation_audit['raw_filepath'].astype(str).ne('')
].copy()

print('RAW ODIR images:', len(raw_df))

raw_predictions, raw_result = evaluate_external(
    raw_df,
    'raw_filepath',
    'RAW_ODIR'
)

raw_predictions.to_csv(
    OUT / 'ODIR_RAW_ZeroShot_Predictions.csv',
    index=False
)

print('\nRAW RESULT:')
display(pd.DataFrame([raw_result]))

print('\n✅ CELL 9 COMPLETE')

In [ ]:
# ============================================================
# CELL 10 — ZERO-SHOT PREPROCESSED ODIR
# ============================================================

pre_df = representation_audit[
    representation_audit['preprocessed_filepath'].astype(str).ne('')
].copy()

print('PREPROCESSED ODIR images:', len(pre_df))

pre_predictions, pre_result = evaluate_external(
    pre_df,
    'preprocessed_filepath',
    'PREPROCESSED_ODIR'
)

pre_predictions.to_csv(
    OUT / 'ODIR_PREPROCESSED_ZeroShot_Predictions.csv',
    index=False
)

print('\nPREPROCESSED RESULT:')
display(pd.DataFrame([pre_result]))

print('\n✅ CELL 10 COMPLETE')

In [ ]:
# ============================================================
# CELL 11 — PAIRED SAME-FILENAME RAW vs PREPROCESSED
# ============================================================

paired_rows = []

if len(common):
    common_raw_pred, common_raw_result = evaluate_external(
        common.copy(),
        'raw_filepath',
        'COMMON_RAW'
    )

    common_pre_pred, common_pre_result = evaluate_external(
        common.copy(),
        'preprocessed_filepath',
        'COMMON_PREPROCESSED'
    )

    paired_rows += [
        common_raw_result,
        common_pre_result
    ]

    agreement = float(
        np.mean(
            common_raw_pred['y_pred_binary'].to_numpy()
            ==
            common_pre_pred['y_pred_binary'].to_numpy()
        )
    )

    score_corr = float(
        np.corrcoef(
            common_raw_pred['Clinical_Cataract_Score'].to_numpy(),
            common_pre_pred['Clinical_Cataract_Score'].to_numpy()
        )[0, 1]
    )

    print('Prediction agreement:', agreement)
    print('Score correlation:', score_corr)

paired_df = pd.DataFrame(paired_rows)

paired_df.to_csv(
    OUT / 'ODIR_Paired_Raw_Preprocessed_Metrics.csv',
    index=False
)

if len(paired_df):
    display(paired_df)

print('\n✅ CELL 11 COMPLETE')

In [ ]:
# ============================================================
# CELL 12 — PATIENT-CLUSTER BOOTSTRAP 95% CIs
# ============================================================

def patient_cluster_bootstrap(pred_df, n_boot=5000, seed=42):
    rng = np.random.default_rng(seed)

    patients = pred_df['patient_id'].astype(str).unique()
    pid_arr = pred_df['patient_id'].astype(str).to_numpy()

    patient_indices = {
        pid: np.where(pid_arr == pid)[0]
        for pid in patients
    }

    y = pred_df['y_true_binary'].to_numpy(dtype=int)
    pred = pred_df['y_pred_binary'].to_numpy(dtype=int)
    score = pred_df['Clinical_Cataract_Score'].to_numpy(dtype=float)

    vals = {
        'Accuracy': [],
        'Sensitivity': [],
        'Specificity': [],
        'AUC': []
    }

    for _ in range(n_boot):
        sampled = rng.choice(
            patients,
            size=len(patients),
            replace=True
        )

        idx = np.concatenate([
            patient_indices[p]
            for p in sampled
        ])

        yy = y[idx]
        pp = pred[idx]
        ss = score[idx]

        if len(np.unique(yy)) < 2:
            continue

        m = binary_metrics(yy, pp, ss)

        for k in vals:
            vals[k].append(m[k])

    out = {}

    for k, arr in vals.items():
        arr = np.asarray(arr, dtype=float)
        arr = arr[np.isfinite(arr)]

        out[k + '_CI_Low'] = float(np.percentile(arr, 2.5))
        out[k + '_CI_High'] = float(np.percentile(arr, 97.5))

    return out

bootstrap_rows = []

for name, pred_df, result in [
    ('RAW_ODIR', raw_predictions, raw_result),
    ('PREPROCESSED_ODIR', pre_predictions, pre_result)
]:
    if len(pred_df) == 0:
        continue

    ci = patient_cluster_bootstrap(
        pred_df,
        n_boot=BOOTSTRAPS,
        seed=SEED
    )

    bootstrap_rows.append({
        **result,
        **ci,
        'Bootstrap_Replicates': BOOTSTRAPS,
        'Bootstrap_Unit': 'patient_id'
    })

bootstrap_df = pd.DataFrame(bootstrap_rows)

bootstrap_df.to_csv(
    OUT / 'ODIR_Final_Representations_With_95CI.csv',
    index=False
)

display(bootstrap_df)

print('\n✅ CELL 12 COMPLETE')

In [ ]:
# ============================================================
# CELL 13 — HISTORICAL vs CORRECTED COMPARISON
# ============================================================

comparison_rows = [{
    'Source': 'Historical_saved_predictions',
    'N': int(len(old_y_true)),
    'Accuracy': float(np.mean(old_y_true == old_y_pred)),
    'Sensitivity': np.nan,
    'Specificity': np.nan,
    'AUC': np.nan,
    'Notes': 'Direct accuracy from existing ODIR5K_y_true/y_pred arrays'
}]

if old_binary_result is not None:
    comparison_rows.append({
        'Source': 'Historical_saved_arrays_clinical_recompute',
        'N': old_binary_result['N'],
        'Accuracy': old_binary_result['Accuracy'],
        'Sensitivity': old_binary_result['Sensitivity'],
        'Specificity': old_binary_result['Specificity'],
        'AUC': old_binary_result['AUC'],
        'Notes': 'Assumes saved classes 0=Cataract, 1=Normal'
    })

for result in [raw_result, pre_result]:
    if result:
        comparison_rows.append({
            'Source': result['Representation'],
            'N': result['N'],
            'Accuracy': result['Accuracy'],
            'Sensitivity': result['Sensitivity'],
            'Specificity': result['Specificity'],
            'AUC': result['AUC'],
            'Notes': (
                'True zero-shot; no ODIR training; '
                'strict image-level labels'
            )
        })

comparison_df = pd.DataFrame(comparison_rows)

comparison_df.to_csv(
    OUT / 'ODIR_Historical_vs_Corrected_Comparison.csv',
    index=False
)

print('========================================')
print('ODIR PROTOCOL COMPARISON')
print('========================================')
display(comparison_df)

print('\nStrict image-level N:', len(strict_unique))
print('Historical saved N:', len(old_y_true))
print('Same N:', len(strict_unique) == len(old_y_true))

print('\n✅ CELL 13 COMPLETE')

In [ ]:
# ============================================================
# CELL 14 — AUTOMATIC FINAL VERDICT
# ============================================================

def pct(x):
    return f'{100*x:.2f}%' if pd.notna(x) else 'NA'

verdict_lines = [
    'ODIR protocol sanity audit completed without model training.',
    f'Historical saved prediction N = {len(old_y_true)}.',
    (
        'Correct strict image-level ODIR Cataract-v-Normal N = '
        f'{len(strict_unique)}.'
    )
]

if raw_result:
    verdict_lines.append(
        'Raw-image zero-shot: '
        f'accuracy {pct(raw_result["Accuracy"])}, '
        f'sensitivity {pct(raw_result["Sensitivity"])}, '
        f'specificity {pct(raw_result["Specificity"])}, '
        f'AUC {raw_result["AUC"]:.4f}.'
    )

if pre_result:
    verdict_lines.append(
        'Preprocessed-image zero-shot: '
        f'accuracy {pct(pre_result["Accuracy"])}, '
        f'sensitivity {pct(pre_result["Sensitivity"])}, '
        f'specificity {pct(pre_result["Specificity"])}, '
        f'AUC {pre_result["AUC"]:.4f}.'
    )

if raw_result and pre_result:
    acc_gap = abs(raw_result['Accuracy'] - pre_result['Accuracy'])
    auc_gap = abs(raw_result['AUC'] - pre_result['AUC'])

    if raw_result['AUC'] < 0.60 and pre_result['AUC'] < 0.60:
        final_verdict = 'BOTH_RAW_AND_PREPROCESSED_ZERO_SHOT_POOR'
        interpretation = (
            'Poor performance persists across both image representations. '
            'Treat ODIR as a domain-shift stress test and a limitation, '
            'not as successful external validation.'
        )
    elif acc_gap > 0.10 or auc_gap > 0.10:
        final_verdict = 'STRONG_REPRESENTATION_SENSITIVITY'
        interpretation = (
            'Raw and preprocessed ODIR results differ materially. '
            'Use the representation matching the original model pipeline '
            'as primary and discuss sensitivity.'
        )
    else:
        final_verdict = 'REPRESENTATIONS_BROADLY_CONSISTENT'
        interpretation = (
            'Raw and preprocessed results are broadly consistent. '
            'Use the raw-image protocol as the primary external evaluation.'
        )
else:
    final_verdict = 'INCOMPLETE_REPRESENTATION_COMPARISON'
    interpretation = (
        'One representation was unavailable; '
        'the protocol sensitivity comparison is incomplete.'
    )

verdict_lines.append('Final protocol verdict: ' + final_verdict)
verdict_lines.append(interpretation)

verdict_text = '\n'.join(verdict_lines)

(OUT / 'ODIR_PROTOCOL_FINAL_VERDICT.txt').write_text(verdict_text)

print('========================================')
print('FINAL ODIR PROTOCOL VERDICT')
print('========================================')
print(verdict_text)

print('\n✅ CELL 14 COMPLETE')

In [ ]:
# ============================================================
# CELL 15 — FINAL COMPLETION CHECK
# ============================================================

required = [
    'Historical_ODIR_Recomputed_Metrics.csv',
    'full_df_Image_Level_Audit.csv',
    'ODIR_Strict_ImageLevel_Cataract_Normal.csv',
    'ODIR_Raw_vs_Preprocessed_File_Audit.csv',
    'Historical_ODIR_Code_Search_Hits.csv',
    'ODIR_Historical_vs_Corrected_Comparison.csv',
    'ODIR_PROTOCOL_FINAL_VERDICT.txt'
]

missing = [
    fn for fn in required
    if not (OUT / fn).exists()
]

if missing:
    print('Missing outputs:')
    for fn in missing:
        print('❌', fn)

    raise RuntimeError(
        'STOP: ODIR protocol sanity audit is incomplete.'
    )

(OUT / 'ODIR_PROTOCOL_SANITY_AUDIT_DONE.txt').write_text(
    'ODIR protocol sanity audit completed.\n'
    'No model training performed.\n'
)

print('========================================')
print('✅ ODIR PROTOCOL SANITY AUDIT COMPLETE')
print('========================================')
print('\nSaved to:')
print(OUT)
print('\nNEXT: Upload this executed notebook to ChatGPT.')